# 9.8 XGBoost - Deep Dive

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 9: Ensemble Learning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-09-ensemble-learning/09.08_xgboost-deep-dive.ipynb)

Companion code for Section 9.8 of the book. Run the cells in order from top to bottom.

**Example 9.8.1** · bash

```bash
import xgboost as xgb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
from sklearn.preprocessing import StandardScaler
import warnings; warnings.filterwarnings('ignore')

data = load_breast_cancer()
X, y = data.data, data.target
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# ── XGBoost with early stopping ───────────────────────────────────
xgb_model = xgb.XGBClassifier(
    n_estimators       = 1000,     # Large - early stopping will find optimal
    learning_rate      = 0.05,
    max_depth          = 4,
    subsample          = 0.8,      # Row subsampling
    colsample_bytree   = 0.8,      # Column subsampling per tree
    colsample_bylevel  = 0.8,      # Column subsampling per level
    gamma              = 0.1,      # Minimum gain to split
    reg_alpha          = 0.1,      # L1 weight penalty
    reg_lambda         = 1.0,      # L2 weight penalty
    min_child_weight   = 3,        # Min Hessian sum per leaf
    eval_metric        = 'logloss',
    early_stopping_rounds = 30,
    use_label_encoder  = False,
    random_state       = 42,
    n_jobs             = -1,
    verbosity          = 0
)

xgb_model.fit(
    X_tr, y_tr,
    eval_set=[(X_tr, y_tr), (X_te, y_te)],
    verbose=False
)

print(f'Best iteration: {xgb_model.best_iteration}')
print(f'Train accuracy: {xgb_model.score(X_tr, y_tr):.4f}')
print(f'Test accuracy:  {xgb_model.score(X_te, y_te):.4f}')
print(f'Test AUC:       {roc_auc_score(y_te, xgb_model.predict_proba(X_te)[:,1]):.4f}')

# ── Hyperparameter tuning with RandomizedSearchCV ─────────────────
param_dist = {
    'n_estimators':       [100, 200, 300],
    'max_depth':          [3, 4, 5, 6],
    'learning_rate':      [0.01, 0.05, 0.1, 0.2],
    'subsample':          [0.6, 0.7, 0.8, 0.9],
    'colsample_bytree':   [0.6, 0.7, 0.8, 0.9],
    'gamma':              [0, 0.1, 0.2, 0.5],
    'reg_alpha':          [0, 0.1, 0.5, 1.0],
    'reg_lambda':         [0.5, 1.0, 2.0, 5.0],
    'min_child_weight':   [1, 3, 5],
}

xgb_base = xgb.XGBClassifier(eval_metric='logloss', use_label_encoder=False,
                               random_state=42, verbosity=0, n_jobs=-1)
rscv = RandomizedSearchCV(
    xgb_base, param_dist, n_iter=40, cv=5,
    scoring='roc_auc', n_jobs=-1, random_state=42
)
rscv.fit(X_tr, y_tr)

print(f'\nBest params: {rscv.best_params_}')
print(f'Best CV AUC: {rscv.best_score_:.4f}')
print(f'Test AUC:    {roc_auc_score(y_te, rscv.predict_proba(X_te)[:,1]):.4f}')

# ── Learning curves ───────────────────────────────────────────────
results = xgb_model.evals_result()
train_logloss = results['validation_0']['logloss']
test_logloss  = results['validation_1']['logloss']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('XGBoost Deep Dive Analysis', fontsize=14, fontweight='bold')

# Learning curves
ax = axes[0]
rounds = range(1, len(train_logloss)+1)
ax.plot(rounds, train_logloss, 'b-', lw=2, alpha=0.8, label='Train')
ax.plot(rounds, test_logloss,  'r-', lw=2, alpha=0.8, label='Test')
ax.axvline(xgb_model.best_iteration, color='green', ls='--', lw=2,
          label=f'Best: {xgb_model.best_iteration}')
ax.set_xlabel('Boosting Rounds'); ax.set_ylabel('Log-Loss')
ax.set_title('Early Stopping: Learning Curves'); ax.legend(); ax.grid(alpha=0.3)

# Feature importance (gain)
ax = axes[1]
imp = pd.Series(xgb_model.get_booster().get_score(importance_type='gain'))
imp.index = [data.feature_names[int(f[1:])] for f in imp.index]
imp = imp.sort_values(ascending=True).tail(15)
ax.barh(imp.index, imp.values, color='#FF6F00', edgecolor='white')
ax.set_xlabel('Gain'); ax.set_title('XGBoost Feature Importance (Gain)')
ax.grid(alpha=0.3, axis='x')

# Hyperparameter comparison
ax = axes[2]
cv_results = pd.DataFrame(rscv.cv_results_)
cv_results = cv_results.sort_values('rank_test_score')
ax.scatter(range(len(cv_results)), cv_results['mean_test_score'],
          c=cv_results['rank_test_score'], cmap='RdYlGn_r', s=30, alpha=0.7)
ax.axhline(rscv.best_score_, color='red', ls='--', lw=2, label=f'Best={rscv.best_score_:.4f}')
ax.set_xlabel('Hyperparameter Configuration (ranked)'); ax.set_ylabel('CV AUC')
ax.set_title('RandomizedSearchCV Results\n(40 configurations)')
ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('xgboost_deep_dive.png', dpi=150, bbox_inches='tight')
plt.show()

# ── XGBoost parameter reference ───────────────────────────────────
print('\n=== XGBoost Parameter Reference ===')
params_ref = {
    'Learning':            {'learning_rate':'0.05-0.3; smaller=more trees needed', 'n_estimators':'use early_stopping_rounds'},
    'Tree Structure':      {'max_depth':'3-6; deeper=more complex; 6 is often max', 'min_child_weight':'1-10; larger=more
conservative'},
    'Sampling':            {'subsample':'0.6-0.9; reduces overfitting', 'colsample_bytree':'0.6-0.9; like RF feature sampling'},
    'Regularisation':      {'gamma':'0-0.5; min gain to split', 'reg_alpha':'L1 penalty', 'reg_lambda':'L2 penalty; 1.0 default'},
    'Class Imbalance':     {'scale_pos_weight':'neg/pos count; for imbalanced data'},
}
for category, params in params_ref.items():
    print(f'  {category}:')
    for param, desc in params.items():
        print(f'    {param:<20}: {desc}')
```